# Total Variation Image Denoising via Subgradient Descent

This notebook demonstrates **total variation (TV) regularized image denoising** using first-order optimization in PyTorch. We formulate denoising as a convex optimization problem and solve it with subgradient descent, illustrating how nonsmooth regularization can effectively remove noise while preserving sharp edges. By the end of this notebook you will understand the TV denoising formulation, its connection to subgradient methods, and the role of the regularization parameter $\lambda$.

## The TV Denoising Problem

Given a noisy observation $y$, we seek a clean image $x$ by solving

$$
\min_x \; \frac{1}{2}\|x - y\|_2^2 \;+\; \lambda \, \mathrm{TV}(x),
$$

where the two terms play complementary roles:

| Term | Role |
|---|---|
| $\frac{1}{2}\|x - y\|_2^2$ | **Data fidelity** -- keeps the reconstruction close to the observed (noisy) image. |
| $\lambda\,\mathrm{TV}(x)$ | **Regularization** -- penalizes rapid oscillations in pixel intensity, encouraging smoothness. |

For a 2-D image the **anisotropic total variation** is

$$
\mathrm{TV}(x) = \sum_{i,j} \bigl(|x_{i+1,j} - x_{i,j}| + |x_{i,j+1} - x_{i,j}|\bigr).
$$

### Why does TV regularization promote piecewise-constant solutions?

The TV penalty measures the total amount of "jump" across neighboring pixels. Unlike an $\ell_2$-penalty on differences (which would blur edges), the $\ell_1$-penalty used in TV is **sparsity-promoting**: it drives many pixel-to-pixel differences to exactly zero while allowing a few large jumps to remain. The result is an image that is constant over large regions separated by sharp edges -- precisely the piecewise-constant structure we want.

### Connection to subgradient methods

The absolute-value terms in $\mathrm{TV}(x)$ make the objective **nonsmooth**: the gradient does not exist at points where $x_{i+1,j} = x_{i,j}$. Standard gradient descent therefore cannot be applied directly. Instead, we use a **subgradient method**. At any point the subgradient of $|u|$ is

$$
\partial |u| = \begin{cases} \{+1\} & u > 0,\\ [-1, +1] & u = 0,\\ \{-1\} & u < 0. \end{cases}
$$

In practice, PyTorch's `torch.abs` computes a subgradient automatically via its backward pass (returning $\mathrm{sign}(u)$, with $0$ at the origin). Combined with a decaying step size, this gives a valid subgradient descent scheme.

### The role of $\lambda$

The parameter $\lambda \ge 0$ controls the fidelity--smoothness trade-off:

- **$\lambda$ too small**: the data-fidelity term dominates and the reconstruction stays noisy.
- **$\lambda$ too large**: the TV term dominates, over-smoothing the image and erasing fine details.
- **$\lambda$ well-chosen**: noise is suppressed while meaningful edges are preserved.

## Implementation

Below we implement the TV denoising pipeline in PyTorch:

1. **`total_variation_loss`** computes $\mathrm{TV}(x)$ by summing absolute horizontal and vertical pixel differences.
2. **`combined_loss`** returns $\frac{1}{2}\|x - y\|_2^2 + \lambda\,\mathrm{TV}(x)$.
3. We initialize the denoised image as a copy of the noisy input and iteratively update it with `torch.optim.SGD`. A `StepLR` scheduler decays the learning rate every 50 iterations (by a factor of 0.5), which is important for subgradient methods where a diminishing step size is needed for convergence.
4. After each update we clamp pixel values to $[0,1]$ to maintain a valid image.

In [ ]:
import torch
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from skimage.data import camera
from skimage.util import random_noise


def total_variation_loss(x):
    """Anisotropic TV: sum of absolute horizontal and vertical differences."""
    dx = torch.abs(x[:, :, 1:, :] - x[:, :, :-1, :])  # |x_{i+1,j} - x_{i,j}|
    dy = torch.abs(x[:, :, :, 1:] - x[:, :, :, :-1])  # |x_{i,j+1} - x_{i,j}|
    return torch.sum(dx) + torch.sum(dy)


def combined_loss(denoised, noisy, lambda_tv):
    """Objective: MSE data fidelity + lambda * TV regularization."""
    mse = torch.nn.functional.mse_loss(denoised, noisy)
    tv = total_variation_loss(denoised)
    return mse + lambda_tv * tv


def generate_noisy_image():
    """Load the 'camera' test image and add Gaussian noise."""
    image = camera().astype(np.float32) / 255.0
    noisy_image = random_noise(image, mode='gaussian', var=1)
    return torch.tensor(noisy_image, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # shape (1,1,H,W)


def visualize_denoising(original, noisy, denoised, iteration):
    """Side-by-side comparison of original, noisy, and denoised images."""
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(original[0, 0].cpu().numpy(), cmap='gray')
    axs[0].set_title('Original Image')
    axs[0].axis('off')
    axs[1].imshow(noisy[0, 0].cpu().numpy(), cmap='gray')
    axs[1].set_title('Noisy Image')
    axs[1].axis('off')
    axs[2].imshow(denoised[0, 0].cpu().detach().numpy(), cmap='gray')
    axs[2].set_title(f'Denoised Image (TV)\nIteration: {iteration}')
    axs[2].axis('off')
    plt.tight_layout()
    plt.show()


def tv_denoising_demo():
    # --- Hyperparameters ---
    lambda_tv = 0.05       # TV regularization weight
    initial_lr = 0.1       # initial step size for SGD
    num_iterations = 500

    # --- Load data ---
    original_image = camera().astype(np.float32) / 255.0
    noisy_image = generate_noisy_image()

    # Initialize denoised image as a copy of the noisy input (warm start)
    denoised_image = noisy_image.clone().requires_grad_(True)

    # SGD optimizer acts as our subgradient descent step
    optimizer = optim.SGD([denoised_image], lr=initial_lr)

    # Decay the step size by 0.5 every 50 iterations (needed for subgradient convergence)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)

    # --- Optimization loop ---
    for step in range(1, num_iterations + 1):
        optimizer.zero_grad()
        loss = combined_loss(denoised_image, noisy_image, lambda_tv)
        loss.backward()            # computes subgradients via autograd
        optimizer.step()

        # Project pixel values back to valid range [0, 1]
        with torch.no_grad():
            denoised_image.clamp_(0, 1)

        scheduler.step()

        if step % 20 == 0 or step == 1:
            current_lr = scheduler.get_last_lr()[0]
            mse = torch.nn.functional.mse_loss(denoised_image, noisy_image).item()
            tv = total_variation_loss(denoised_image).item()
            print(f"Step {step:03d}, Loss: {loss.item():.4f}, MSE: {mse:.4f}, TV: {tv:.4f}, LR: {current_lr:.5f}")

        if step % 100 == 0:
            visualize_denoising(
                torch.tensor(original_image).unsqueeze(0).unsqueeze(0),
                noisy_image, denoised_image, step
            )

    # Final visualization
    visualize_denoising(
        torch.tensor(original_image).unsqueeze(0).unsqueeze(0),
        noisy_image, denoised_image, num_iterations
    )


tv_denoising_demo()

## Summary

**Key takeaways from this notebook:**

- **TV denoising** solves $\min_x \frac{1}{2}\|x-y\|_2^2 + \lambda\,\mathrm{TV}(x)$, balancing fidelity to the noisy observation against smoothness of the reconstruction.
- The TV penalty uses an $\ell_1$-norm on finite differences, which promotes **piecewise-constant** solutions -- flat regions separated by sharp edges -- unlike $\ell_2$-based smoothing which blurs edges.
- Because $\mathrm{TV}(x)$ is **nonsmooth**, we rely on subgradient descent rather than standard gradient descent. PyTorch's autograd computes valid subgradients through `torch.abs`.
- A **decaying step size** (here via `StepLR`) is essential for subgradient methods to converge, since constant step sizes only guarantee convergence to a neighborhood of the optimum.
- The regularization parameter $\lambda$ controls the noise--detail trade-off: too small leaves noise, too large erases texture. Tuning $\lambda$ is critical in practice.